# Run

In [ ]:
import os
import pandas as pd
from BERTopic_model import run_BERTopic_model

# Dataloading
df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])
docs = [doc.replace('\xa0', '') for doc in docs]

# hyperparameters setting
base_param_dic = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "n_neighbors": 15,
    "n_components": 10,
    "min_dist": 0.0,
    "min_cluster_size": 20,
    "min_df": 3,
    "max_df": 1.0,
    "ngram_range": (1, 3),
    "top_n_words": 5,
    "seed": 0
}

all_param_dic = {
    "n_neighbors": [5, 30],
    "n_components": [5, 15],
    "min_dist": [0.1, 0.5],
    "min_cluster_size": [10, 30],
    "min_df": [0.0, 10],
    "max_df": [0.5, 0.8],
    "ngram_range": [(1, 1), (1, 5)],
    "top_n_words": [10, 20]
}

# run
for key, values in all_param_dic.items():
    for value in values:
        run_name = f'{"bertopic"}_{key}-{str(value)}'
        param_dic = base_param_dic.copy()
        param_dic[key] = value
        topic_model = run_BERTopic_model(param_dic, docs)
        topic_model.save("../../results/NLP/robustness_check", serialization="safetensors", save_ctfidf=True, save_embedding_model="all-MiniLM-L6-v2")

## Correlation

Compute correlation between model knowledge & different BERTopic model

In [ ]:
def compute_topic_corr(topic_model, docs, var):

    topic_distr, _ = topic_model.approximate_distribution(docs)
    df_stat = pd.read_csv("../../data/NLP_data_stake.csv")

    for topic_n in range(len(topic_distr[0,:])):
        topic_name = "topic_" + str(topic_n)
        df_stat[topic_name] = topic_distr[:, topic_n]

    df_stat["assigned_topic"] = topic_model.topics_
    return df_stat["pl2_understanding"].corr(df_stat[var])
run_name = []
corr = []
n = []

base_model = BERTopic.load("../../results/NLP/BERTopic_model", embedding_model=embedding_model)

corr.append(compute_topic_corr(base_model, docs, "topic_1"))
run_name.append("base")
n.append(base_model.topics_.count(1))

results_dir = "../../results/NLP/robustness_check"
robustness_models = [f for f in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, f))]

# arbitrary topic number, depends on the specific model, decision made by human inspection
topic_n = [2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 1, 2, 1, 2, 2]

i = 0
for model_name in robustness_models:
    topic_model = BERTopic.load("../../results/NLP/robustness_check/" + model_name, embedding_model=embedding_model)
    corr.append(compute_topic_corr(topic_model, docs, "topic_" + str(topic_n[i])))
    run_name.append(model_name)
    n.append(topic_model.topics_.count(topic_n[i]))
    i += 1


Saves results

In [ ]:
data = pd.DataFrame({
    "run_name": run_name,
    "corr": corr,
    "n": n
})
data.to_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "robustness_check", "results.csv"), header=True, index=False)

,run_name,corr,n
0,base,-0.267919,197
1,bertopic_max_df-0.5,0.481482,144
2,bertopic_max_df-0.8,0.497067,144
3,bertopic_min_cluster_size-10,0.599498,104
